# 🏡 Regression: Smart kNN & 2-Stage Stratification

### 📦 Installation
```bash
pip install -e .
```

### 🎯 Overview
`FastKMeansRegressor` is the ultimate scalable substitution for the classic `KNeighborsRegressor`.

It utilizes **2-Stage Target-Feature Stratification**:
1. First, the continuous target variables ($Y$) are clustered into `k_targets` geographic buckets.
2. Second, `k_features` prototypes are spawned inside each bucket. 

Crucially, the very first prototype inside any bucket is mathematically forced to be the stratum's global mean. This ensures that extreme outliers (e.g., extremely expensive houses) have dedicated prototypes and are never "averaged out" by the majority. We evaluate this on the **California Housing** dataset.

In [ ]:
import time
import pandas as pd
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from IPython.display import display

from fast_kmeans import FastKMeansRegressor

# 1. Load Regression Dataset (California Housing)
X_np, y_np = fetch_california_housing(return_X_y=True)

# Standardizing inputs and outputs is strictly required for metric learning
x_scaler, y_scaler = StandardScaler(), StandardScaler()
X_scaled = x_scaler.fit_transform(X_np)
y_scaled = y_scaler.fit_transform(y_np.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.1, random_state=42)
y_test_unscaled = y_scaler.inverse_transform(y_test) # Keep unscaled targets for honest metric calculation

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

print("🚀 Benchmarking FastKMeansRegressor vs Standard KNeighborsRegressor...\n")

# --- SKLEARN KNN REGRESSOR ---
knn = KNeighborsRegressor(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train, y_train)

start = time.time()
knn_preds_scaled = knn.predict(X_test)
knn_inf_time = time.time() - start

knn_preds = y_scaler.inverse_transform(knn_preds_scaled)
knn_r2 = r2_score(y_test_unscaled, knn_preds)

# --- FAST-KMEANS REGRESSOR ---
fkm_reg = FastKMeansRegressor(
    k_targets=20,       # Number of Y-stratas (buckets)
    k_features=20,      # Feature prototypes per bucket
    distance='euclidean',
    target_distance='euclidean', 
    target_assignment='softmax_scaled', 
    soft_type='softmax_scaled', # Exponentiated IDW
    top_k=5,
    batch_size=1024,
    diversity_reg=0.01
)
fkm_reg.fit(X_train_t, y_train_t, max_iters=300, verbose=True)
fkm_reg.finetune(X_train_t, y_train_t, epochs=100, eval_fraction=0.2, lr=0.1, verbose=True)

start = time.time()
fkm_preds_scaled = fkm_reg.predict(X_test_t).cpu().numpy()
fkm_inf_time = time.time() - start

fkm_preds = y_scaler.inverse_transform(fkm_preds_scaled)
fkm_r2 = r2_score(y_test_unscaled, fkm_preds)

# 2. Build Comparison Table
results = pd.DataFrame({
    "Algorithm": ["Sklearn KNN Regressor", "FastKMeansRegressor (Smart KNN)"],
    "Routing Mechanism": ["Uniform / Inverse Distance", "Differentiable Softmax-Scaled IDW"],
    "Memory Footprint": ["Stores all 18,576 samples", "Compresses into 400 Topologies"],
    "R^2 Score": [f"{knn_r2:.4f}", f"{fkm_r2:.4f}"],
    "Inference Time (s)": [f"{knn_inf_time:.4f}", f"{fkm_inf_time:.4f}"]
})

display(results.style.set_caption("Smart KNN Regression Benchmark (California Housing)")
        .set_properties(**{'text-align': 'center'}))

🚀 Benchmarking FastKMeansRegressor vs Standard KNeighborsRegressor...



Fit (EMA Topology):   0%|          | 0/300 [00:00<?, ?it/s]

Finetune (Gradient):   0%|          | 0/100 [00:00<?, ?it/s]

,Algorithm,Routing Mechanism,Memory Footprint,R^2 Score,Inference Time (s)
0,Sklearn KNN Regressor,Uniform / Inverse Distance,"Stores all 18,576 samples",0.6718,0.0926
1,FastKMeansRegressor (Smart KNN),Differentiable Softmax-Scaled IDW,Compresses into 400 Topologies,0.5891,0.0072
